# Estructura y calidad de la serie RB-IA-2191

**Dataset:** RB-IA-2191 · **Cuaderno:** 01 · **Tema:** geometría de la tabla y continuidad temporal

## Introducción

BES-jun organiza varios pozos en hojas de diseño. RB-1521 es una serie diaria larga sin código de pozo interno. RB-IA-2191 también es una serie temporal, pero más corta y en formato `.xlsx`. Tratarla como un inventario multi-pozo o como una réplica de RB-1521 sería un error de lectura.

## Objetivos

Describir la geometría real del libro, comprobar la continuidad de las fechas y separar columnas constantes de columnas variables.

## Metodología

`openpyxl` con `data_only=True` entrega el valor almacenado. Las celdas vacías permanecen como ausencia. La continuidad se juzga por la diferencia, en días, entre fechas consecutivas. No se imputan huecos.

## Fundamento

Una serie diaria continua permite fijar un día de referencia. Una serie con huecos obligaría a decidir si el hueco es parada o registro perdido; esa decisión no se toma aquí.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from collections import Counter
from rbia2191_lectura import fecha_iso, leer_serie
serie = leer_serie()
fechas = [fila["FECHA"] for fila in serie]
deltas = Counter((fechas[i] - fechas[i-1]).days for i in range(1, len(fechas)))
print("filas", len(serie))
print("desde", fecha_iso(fechas[0]), "hasta", fecha_iso(fechas[-1]))
print("deltas", dict(deltas))
constantes = {}
variables = {}
for clave in serie[0]:
    if clave == "FECHA":
        continue
    presentes = [fila[clave] for fila in serie if fila[clave] is not None]
    unicos = set(presentes)
    if len(unicos) == 1:
        constantes[clave] = presentes[0]
    else:
        variables[clave] = len(unicos)
print("constantes", sorted(constantes))
print("variables (n únicos)", variables)


filas 28
desde 2020-05-02 hasta 2020-05-29
deltas {1: 27}
constantes ['API', 'Amp_m', 'BFPD', 'BOPD', 'BSW', 'BWPD', 'FP', 'GE_W', 'H', 'HZ', 'PIP', 'PWF', 'P_Sensor', 'P_cas', 'T_Yac', 'Visco_f', 'Volt_m']
variables (n únicos) {'Sum': 19, 'THP': 2, 'P_SUT': 18, 'Pbomba': 19, 'Eff_b': 6, 'Porc_Load': 18, 'Eff_m': 18}


## Resultados

Hay una sola hoja, `Hoja1`, con 28 filas de datos (2020-05-02 a 2020-05-29) y 25 columnas. El salto entre fechas consecutivas es siempre un día.

Permanecen constantes BFPD, BWPD, BOPD, HZ, P_Sensor, H, BSW, PWF, Volt_m, Amp_m, API, T_Yac, GE_W, P_cas, Visco_f, FP y PIP. Varían Sum, THP, P_SUT, Pbomba, Eff_b, Porc_Load y Eff_m.

## Interpretación

El libro describe una serie de un solo conjunto operativo. El nombre `RB-IA-2191` identifica el expediente; el archivo no trae otro código de pozo. No hay columna de marca de bomba.

## Restricción

Que una columna sea constante no demuestra que sea un parámetro de diseño independiente. Solo demuestra que el archivo la repite. Las columnas variables sin fórmula visible no se recalculan en el motor.

## Conclusión

La estructura útil es una tabla diaria 28×25, continua, con un bloque de caudales constantes y un bloque de magnitudes derivadas almacenadas sin expresión.
